# Data Visualization with Matplotlib and Seaborn
**GDG on Campus — AI/ML Team**  
**COMSATS University, Attock Campus**

This notebook completes all **14 tasks** in the weekly task sheet, in the required order. Each visualization is followed by a short insight.

> **Note:** Seaborn's sample datasets are downloaded/cached by `sns.load_dataset()` when needed, so the first run may require an internet connection.

## Setup and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the sample datasets specified in the task sheet.
tips = sns.load_dataset("tips")
flights = sns.load_dataset("flights")
mpg = sns.load_dataset("mpg")
penguins = sns.load_dataset("penguins")
titanic = sns.load_dataset("titanic")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

print("Datasets loaded:")
print(f"tips: {tips.shape}, flights: {flights.shape}, mpg: {mpg.shape}")
print(f"penguins: {penguins.shape}, titanic: {titanic.shape}")

# Part 1: Matplotlib Tasks

## Task 1 — Line plot (flights)
Plot monthly passengers for one year, with title, axis labels, markers, and grid.

In [ ]:
# Use 1958 as the example year and keep months in calendar order.
year_data = flights[flights["year"] == 1958]

fig, ax = plt.subplots()
ax.plot(year_data["month"], year_data["passengers"], marker="o", linewidth=2,
        label="Passengers")
ax.set_title("Monthly Air Passengers in 1958")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Passengers")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
Passenger numbers rise through the middle of the year and fall toward the end. This suggests a seasonal pattern, with the busiest travel period occurring in summer.

## Task 2 — Bar chart (tips)
Plot average `total_bill` by day, add values above bars, and use custom colors.

In [ ]:
avg_bill_by_day = tips.groupby("day", observed=False)["total_bill"].mean().reindex(
    ["Thur", "Fri", "Sat", "Sun"]
)
bar_colors = ["#6BAED6", "#74C476", "#FD8D3C", "#9E9AC8"]

fig, ax = plt.subplots()
bars = ax.bar(avg_bill_by_day.index, avg_bill_by_day.values, color=bar_colors,
              edgecolor="black", label="Average bill")
ax.bar_label(bars, fmt="$%.2f", padding=3)
ax.set_title("Average Total Bill by Day")
ax.set_xlabel("Day")
ax.set_ylabel("Average Total Bill ($)")
ax.legend()
ax.set_ylim(0, avg_bill_by_day.max() * 1.18)
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
Average bills are generally higher on the weekend than on Thursday or Friday. Sunday and Saturday are useful days to compare because their average bills are among the highest.

## Task 3 — Horizontal bar chart (mpg)
Plot average miles per gallon by cylinder count, sorted from highest to lowest.

In [ ]:
mpg_by_cylinders = (
    mpg.groupby("cylinders", observed=True)["mpg"].mean().sort_values(ascending=False)
)

fig, ax = plt.subplots()
bars = ax.barh(mpg_by_cylinders.index.astype(str), mpg_by_cylinders.values,
               color="#59A14F", edgecolor="black")
ax.bar_label(bars, fmt="%.1f", padding=4)
ax.invert_yaxis()  # Highest average appears at the top.
ax.set_title("Average MPG by Number of Cylinders")
ax.set_xlabel("Average Miles per Gallon (MPG)")
ax.set_ylabel("Cylinders")
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
Cars with fewer cylinders tend to have higher average fuel economy. The result shows a general trade-off between engine size and miles per gallon in this dataset.

## Task 4 — Three histograms (tips)
Show `total_bill` with three different bin sizes side by side and mark the mean and median.

In [ ]:
bill_values = tips["total_bill"].dropna()
bill_mean = bill_values.mean()
bill_median = bill_values.median()
bin_options = [8, 20, 35]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, bins in zip(axes, bin_options):
    ax.hist(bill_values, bins=bins, color="#4E79A7", edgecolor="white", alpha=0.85)
    ax.axvline(bill_mean, color="#E15759", linestyle="--", linewidth=2,
               label=f"Mean = ${bill_mean:.2f}")
    ax.axvline(bill_median, color="#F28E2B", linestyle=":", linewidth=2,
               label=f"Median = ${bill_median:.2f}")
    ax.set_title(f"{bins} bins")
    ax.set_xlabel("Total bill ($)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Frequency")
fig.suptitle("Distribution of Total Bills with Different Bin Sizes", y=1.03)
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
Most bills are concentrated at lower amounts, while a smaller number of large bills create a right-skewed distribution. More bins show finer detail, while fewer bins make the overall shape easier to see; the mean is typically above the median in a right-skewed distribution.

## Task 5 — Scatter plot (mpg)
Plot weight against MPG, color points by origin, size points by horsepower, and include a legend.

In [ ]:
scatter_data = mpg.dropna(subset=["weight", "mpg", "origin", "horsepower"]).copy()
origins = sorted(scatter_data["origin"].unique())
origin_colors = {"usa": "#E15759", "europe": "#59A14F", "japan": "#4E79A7"}

fig, ax = plt.subplots()
for origin in origins:
    subset = scatter_data[scatter_data["origin"] == origin]
    sizes = 18 + (subset["horsepower"] - scatter_data["horsepower"].min()) / (
        scatter_data["horsepower"].max() - scatter_data["horsepower"].min()
    ) * 100
    ax.scatter(subset["weight"], subset["mpg"], s=sizes,
               color=origin_colors.get(str(origin).lower(), "#9C755F"),
               alpha=0.65, edgecolors="white", linewidths=0.4, label=str(origin).title())
ax.set_title("Vehicle Weight vs. Fuel Economy")
ax.set_xlabel("Weight (lbs)")
ax.set_ylabel("Miles per Gallon (MPG)")
ax.legend(title="Origin")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
The plot shows a clear negative relationship: heavier vehicles generally have lower MPG. The colors compare vehicle origins, and larger points represent vehicles with more horsepower.

## Task 6 — Donut chart (tips)
Show the smoker vs. non-smoker share with percentage labels.

In [ ]:
smoker_counts = tips["smoker"].value_counts().reindex(["Yes", "No"]).dropna()
fig, ax = plt.subplots()
wedges, texts, autotexts = ax.pie(
    smoker_counts.values,
    labels=["Smoker", "Non-smoker"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#F28E2B", "#4E79A7"],
    wedgeprops={"width": 0.42, "edgecolor": "white"},
    pctdistance=0.78
)
ax.set_title("Share of Bills by Smoker Status")
ax.axis("equal")
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
The chart compares the number of records for smokers and non-smokers in the tips dataset. The percentages describe the dataset's observations, not the smoking rate of a wider population.

## Task 7 — Mini dashboard
Create a 2 × 2 figure combining four plots and save it as a 300-dpi PNG file.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Average bill by day
avg_bill_by_day.plot(kind="bar", ax=axes[0, 0], color="#4E79A7")
axes[0, 0].set_title("Average Bill by Day")
axes[0, 0].set_xlabel("Day")
axes[0, 0].set_ylabel("Average total bill ($)")
axes[0, 0].tick_params(axis="x", rotation=0)

# 2. Passenger trend in 1958
axes[0, 1].plot(year_data["month"], year_data["passengers"], marker="o",
                color="#F28E2B", label="Passengers")
axes[0, 1].set_title("Monthly Passengers (1958)")
axes[0, 1].set_xlabel("Month")
axes[0, 1].set_ylabel("Passengers")
axes[0, 1].tick_params(axis="x", rotation=35)
axes[0, 1].legend()

# 3. Weight vs MPG
axes[1, 0].scatter(scatter_data["weight"], scatter_data["mpg"],
                   c=scatter_data["horsepower"], cmap="viridis", alpha=0.7)
axes[1, 0].set_title("Weight vs. MPG")
axes[1, 0].set_xlabel("Weight (lbs)")
axes[1, 0].set_ylabel("MPG")

# 4. Distribution of total bills
axes[1, 1].hist(bill_values, bins=20, color="#59A14F", edgecolor="white")
axes[1, 1].set_title("Total Bill Distribution")
axes[1, 1].set_xlabel("Total bill ($)")
axes[1, 1].set_ylabel("Frequency")

fig.suptitle("Data Visualization Mini Dashboard", fontsize=15)
fig.tight_layout()
fig.savefig("matplotlib_mini_dashboard.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: matplotlib_mini_dashboard.png (300 dpi)")

**What does this dashboard tell us?**  
The four panels give a quick overview of dining bills, seasonal air travel, vehicle fuel economy, and bill distribution. Together, they show how different chart types help compare categories, trends, relationships, and distributions.

# Part 2: Seaborn Tasks

## Task 1 — Countplot (titanic)
Plot passenger counts by class, with `sex` as the hue.

In [ ]:
fig, ax = plt.subplots()
sns.countplot(data=titanic, x="class", hue="sex", order=["First", "Second", "Third"],
              palette="Set2", ax=ax)
ax.set_title("Titanic Passengers by Class and Sex")
ax.set_xlabel("Passenger class")
ax.set_ylabel("Number of passengers")
ax.legend(title="Sex")
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
The countplot compares the number of recorded male and female passengers within each class. It makes differences in class composition visible without implying anything about survival.

## Task 2 — Histogram with KDE (penguins)
Plot `flipper_length_mm` for each species using `histplot(kde=True)`.

In [ ]:
penguin_flippers = penguins.dropna(subset=["flipper_length_mm", "species"])
fig, ax = plt.subplots()
sns.histplot(data=penguin_flippers, x="flipper_length_mm", hue="species",
             kde=True, element="step", stat="density", common_norm=False,
             alpha=0.25, ax=ax)
ax.set_title("Penguin Flipper Length by Species")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
The distributions show that flipper length differs by species, although some values overlap. The KDE curves help compare the overall shape and concentration of each species' measurements.

## Task 3 — Box plot vs. violin plot (penguins)
Show `body_mass_g` by species side by side, then compare what each plot reveals.

In [ ]:
penguin_mass = penguins.dropna(subset=["body_mass_g", "species"])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=penguin_mass, x="species", y="body_mass_g", color="#A0CBE8", ax=axes[0])
axes[0].set_title("Box Plot: Body Mass by Species")
axes[0].set_xlabel("Species")
axes[0].set_ylabel("Body mass (g)")

sns.violinplot(data=penguin_mass, x="species", y="body_mass_g", color="#FFBE7D",
               inner="quartile", cut=0, ax=axes[1])
axes[1].set_title("Violin Plot: Body Mass by Species")
axes[1].set_xlabel("Species")
axes[1].set_ylabel("Body mass (g)")
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
The box plot makes the median, quartiles, and potential outliers easier to compare. The violin plot shows the shape and density of the distribution, making it easier to see where body masses are concentrated; neither is universally better, because they emphasize different details.

## Task 4 — Scatterplot (penguins)
Plot bill length against bill depth with `hue=species` and `style=sex`.

In [1]:
bill_data = penguins.dropna(subset=["bill_length_mm", "bill_depth_mm", "species", "sex"])
fig, ax = plt.subplots()
sns.scatterplot(data=bill_data, x="bill_length_mm", y="bill_depth_mm",
                hue="species", style="sex", s=75, alpha=0.8, ax=ax)
ax.set_title("Penguin Bill Length vs. Bill Depth")
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Bill depth (mm)")
ax.legend(title="Species / sex", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

NameError: name 'penguins' is not defined

**What does this chart tell us?**  
Species form partly distinct clusters in bill measurements, which suggests bill dimensions can help distinguish species. The marker styles also let us compare the recorded sexes within those clusters.

## Task 5 — Heatmap (penguins)
Plot the correlation matrix of numeric columns with annotations and a diverging colormap.

In [ ]:
numeric_penguins = penguins.select_dtypes(include="number")
correlation = numeric_penguins.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix of Numeric Penguin Measurements")
plt.tight_layout()
plt.show()

**What does this chart tell us?**  
The heatmap shows the direction and strength of linear relationships between numeric measurements. Positive values indicate that two measurements tend to increase together, while negative values indicate an inverse relationship; correlation alone does not prove causation.

## Task 6 — Pairplot (penguins)
Create a pairplot with `hue=species` and write three observations.

In [ ]:
pairplot_data = penguins.dropna(subset=[
    "species", "bill_length_mm", "bill_depth_mm",
    "flipper_length_mm", "body_mass_g"
])
pair_grid = sns.pairplot(
    pairplot_data,
    vars=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"],
    hue="species",
    diag_kind="hist",
    corner=True,
    plot_kws={"alpha": 0.7, "s": 35}
)
pair_grid.fig.suptitle("Pairwise Relationships in Penguin Measurements", y=1.02)
plt.show()

**Three insights from the pairplot:**

1. Gentoo penguins generally have longer flippers and higher body mass than the other two species.
2. Bill length and bill depth help separate species, though the distributions are not perfectly distinct.
3. Flipper length and body mass show a positive relationship: penguins with longer flippers tend to be heavier.

## Task 7 — lmplot / FacetGrid (tips)
Plot `total_bill` against `tip` with a regression line, split by `time` and `smoker`, and apply a custom Seaborn theme.

In [ ]:
sns.set_theme(style="ticks", context="notebook", palette="colorblind")
lm_grid = sns.lmplot(
    data=tips,
    x="total_bill",
    y="tip",
    hue="smoker",
    col="time",
    height=4.5,
    aspect=1.0,
    scatter_kws={"alpha": 0.65, "s": 35},
    line_kws={"linewidth": 2}
)
lm_grid.set_axis_labels("Total bill ($)", "Tip ($)")
lm_grid.set_titles("Meal time: {col_name}")
lm_grid.fig.suptitle("Tips vs. Total Bill by Meal Time and Smoker Status", y=1.05)
plt.show()

**What does this chart tell us?**  
Tips generally increase as total bills increase, which is reflected by the upward regression lines. The facets compare lunch and dinner, while the colors compare smoker and non-smoker records; these patterns describe the sample and should not be treated as proof of causation.